In [ ]:
import csv
import random
import pathlib as Path

In [ ]:
# Wrapper helpers
def get_sub_ses(dir_raw, verbose=0):
    pt_ids = set()
    subject_session_pairs = []
    
    # Read participants.tsv
    pt_tsv = dir_raw / "participants.tsv"
    with open(pt_tsv, newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        if "participant_id" not in reader.fieldnames:
            raise ValueError("participants.tsv must contain a 'participant_id' column")

        for row in reader:
            sub = row["participant_id"].strip()
            
            if not sub:
                continue
            pt_ids.add(sub)
            
            sub_dir = dir_raw / sub
            sessions_file = sub_dir / f"{sub}_sessions.tsv"
            
            if sessions_file.exists():
                with open(sessions_file, newline="") as sf:
                    sreader = csv.DictReader(sf, delimiter="\t")

                    if "session_id" not in sreader.fieldnames:
                        raise ValueError(f"{sessions_file} must contain a 'session_id' column")

                    for srow in sreader:
                        session_id = srow["session_id"].strip()
                        if session_id:
                            subject_session_pairs.append((sub, session_id))
            else: # No sessions.tsv
                ses_dirs = sorted(
                    d.name for d in sub_dir.glob("ses-*") if d.is_dir()
                )

                if ses_dirs:
                    for ses in ses_dirs:
                        subject_session_pairs.append((sub, ses))
                else:
                    subject_session_pairs.append((sub, None))
                    pt_ids.add(sub)

    if verbose > 0:
        print(f"{len(subject_session_pairs)} subject-session pairs ({len(pt_ids)} unique participants).")
    return subject_session_pairs

def get_todo(params):
    sub_ses_pairs = get_sub_ses(Path.Path(params['root']) / "rawdata")
    if params['test']:
        todo = random.sample(sub_ses_pairs, params['n_test'])
        if params['verbose'] > 0:
            print(f"[PreScrubber] TEST MODE: processing {params['n_test']} subject-session pairs ({todo})")
    else:
        todo = sub_ses_pairs
        if params['verbose'] > 0:
            print(f"[PreScrubber] processing {len(sub_ses_pairs)} subject-session pairs")
    return todo

def mk_txt(todo, out, sep=','):
    lines = []
    for sub, ses in todo:
        line = f"{sub}{sep}{ses}"
        lines.append(line)
    with open(out, 'w') as f:
        f.write("\n".join(lines))
    print(f"[INFO] Saved {len(lines)} lines to {out}")
    return


In [ ]:
# create list file with all sub-ses pairs in dir
params = {
    'root':"/host/verges/tank/data/BIDS_MPN",
    
    'test': False,
    'n_test': 1,
    'verbose': 0,
}

# Save all sub-ses pairs to txt file
todo = get_todo(params)
mk_txt(todo,  "/host/verges/tank/data/daniel/99_varia/MPN/MPN_29jun2026.txt")